# Multi-Hop Retrieval - Train ComplementGenerator (G+D, retrieval_v3)

**Before running:**
1. Settings -> Accelerator -> **T4 GPU**
2. Settings -> Internet -> **On**
3. Google Drive `musique_data/` folder must contain:
   - `musique_ans_v1.0_train.jsonl`
   - `musique_ans_v1.0_dev.jsonl`
4. **(For auto-upload to Drive)** Add-ons -> Secrets -> add secret `RCLONE_CONF`
   = your rclone `[gdrive]` config block (see cell 8).

**What this trains - G+D architecture (fixes v1/v2 collapse):**
- Generator G(A,B): shared BERT + B->A cross-attention + lambda subtraction + complement gate -> 128-dim edge
- Discriminator D(A,e): reconstructs B from A + edge, 40% context dropout on A
- Loss: L_rec + 0.1 x InfoNCE diversity

**Critical gate:** `collapse_sim` should drop below **0.80** by epoch 2 (v1/v2 were stuck >0.95).

**Expected time: ~5-6 hours on T4**

In [ ]:
# [EDIT IF NEEDED]
REPO_URL        = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME       = "multihop-retrieval"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"   # musique_data folder ID
WORK_DIR        = f"/kaggle/working/{REPO_NAME}/retrieval_v3"
UPLOAD_TO_DRIVE = True    # set False to skip auto-upload (then download manually)

In [ ]:
# 1. Verify GPU - must be T4 (sm_70+), not P100 (sm_60)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Settings -> Accelerator -> T4 GPU")
props   = torch.cuda.get_device_properties(0)
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {props.total_memory/1e9:.1f} GB")
print(f"SM   : {props.major}.{props.minor}")
if props.major < 7:
    raise RuntimeError("GPU is P100 (sm_60) - change to T4 GPU")
print("CUDA OK")

In [ ]:
# 2. Clone repo + install dependencies
import os
repo_root = f"/kaggle/working/{REPO_NAME}"
if not os.path.isdir(repo_root):
    os.system(f"git clone {REPO_URL} {repo_root}")
else:
    os.system(f"cd {repo_root} && git pull")
os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())
os.system("pip install -q 'transformers>=4.35.0' 'rank_bm25>=0.2.2' gdown")
print("Dependencies ready")

In [ ]:
# 3. Download MuSiQue data from Google Drive
import os, gdown
download_dir = "/kaggle/working/drive_data"
if not os.path.isdir(download_dir):
    print("Downloading from Google Drive...")
    gdown.download_folder(id=DRIVE_FOLDER_ID, output=download_dir, quiet=False, use_cookies=False)
else:
    print("Drive data already downloaded")
print("\nDownloaded files:")
for f in sorted(os.listdir(download_dir)):
    print(f"  {f:45s}  {os.path.getsize(f'{download_dir}/{f}')/1e6:7.1f} MB")

In [ ]:
# 4. Set up file paths - v3 uses data_loader from retrieval_v2/, symlink data there
import os
drive   = "/kaggle/working/drive_data"
v2_data = f"/kaggle/working/{REPO_NAME}/retrieval_v2/data/musique"
os.makedirs(v2_data,              exist_ok=True)
os.makedirs(f"{WORK_DIR}/models", exist_ok=True)
os.makedirs(f"{WORK_DIR}/cache",  exist_ok=True)
os.makedirs(f"{WORK_DIR}/results",exist_ok=True)
for fname in ["musique_ans_v1.0_train.jsonl", "musique_ans_v1.0_dev.jsonl"]:
    src, dst = f"{drive}/{fname}", f"{v2_data}/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"  {fname}: OK ({os.path.getsize(src)/1e6:.0f} MB)")
print("\nAll paths ready")

In [ ]:
# 5. Smoke test - verify model builds, data loads, one epoch runs
import os
os.chdir(WORK_DIR)
print("=== Smoke test (50 examples, 1 epoch) ===")
os.system("python generator_train.py --smoke")

---
## Train - full 3-epoch run

Watch `collapse_sim` in the logs (every 200 steps + each epoch). Target: < 0.80 by epoch 2.
If stuck > 0.95: collapse (same as v1/v2).

In [ ]:
# 6. Train (full 3-epoch run)
import os
os.chdir(WORK_DIR)
log_file = "/kaggle/working/generator_train.log"
os.system(f"python generator_train.py 2>&1 | tee {log_file}")
print("\nTraining complete")

In [ ]:
# 7. Verify checkpoints + copy generator_best.pt to /kaggle/working/ for manual download
import os, shutil
best  = f"{WORK_DIR}/models/generator_best.pt"
final = f"{WORK_DIR}/models/generator_final.pt"
for path in [best, final]:
    if os.path.exists(path):
        print(f"  {os.path.basename(path)}: {os.path.getsize(path)/1e6:.1f} MB  OK")
    else:
        print(f"  {os.path.basename(path)}: NOT FOUND - check training log")
if os.path.exists(best):
    shutil.copy(best, "/kaggle/working/generator_best.pt")
    print("  Copied to /kaggle/working/generator_best.pt")

---
## 8. Auto-upload to Google Drive (rclone)

**One-time setup on your LOCAL PC:**
1. `winget install Rclone.Rclone` (reopen PowerShell)
2. `rclone config` -> new remote named `gdrive`, storage `drive`, blank client_id/secret,
   scope `1`, auto config `y` (browser auth), shared drive `n`
3. `rclone config file` -> open it -> copy the entire `[gdrive]` block (has `token = {...}`)

**In this notebook:** Add-ons -> Secrets -> add secret named `RCLONE_CONF`,
value = that `[gdrive]` block. The cell below reads it and pushes the model to Drive.

The token is a secret - keep it in Secrets, never paste into a cell.

In [ ]:
# 8. Upload generator_best.pt straight to Drive musique_data/ folder
import os
if UPLOAD_TO_DRIVE:
    try:
        from kaggle_secrets import UserSecretsClient
        conf = UserSecretsClient().get_secret("RCLONE_CONF")
        os.makedirs("/root/.config/rclone", exist_ok=True)
        with open("/root/.config/rclone/rclone.conf", "w") as fh:
            fh.write(conf)
        # install rclone binary
        os.system("curl -s https://downloads.rclone.org/rclone-current-linux-amd64.zip -o /tmp/r.zip")
        os.system("cd /tmp && unzip -qo r.zip && cp rclone-*-linux-amd64/rclone /usr/bin/ && chmod +x /usr/bin/rclone")
        # push to musique_data/ folder
        rc = os.system(
            "rclone copy /kaggle/working/multihop-retrieval/retrieval_v3/models/generator_best.pt "
            f"gdrive: --drive-root-folder-id {DRIVE_FOLDER_ID} -P"
        )
        print("Uploaded to Drive musique_data/" if rc == 0 else f"rclone exited {rc}")
    except Exception as e:
        print("Auto-upload skipped/failed:", e)
        print("Add the RCLONE_CONF secret, or download generator_best.pt from the Output tab.")
else:
    print("UPLOAD_TO_DRIVE=False - download generator_best.pt manually from Output tab.")

---
## 9. ABLATION (one-click) — train with context dropout OFF

This is the key scientific experiment. Trains an identical generator but with
`--context_drop 0.0` (no context dropout). **Expectation: it collapses** —
collapse_sim stays high (>0.9) and at eval FULL falls back to ≈ cosine.

That proves context dropout is the mechanism that makes label-free complement
learning work. Saved as `generator_best_nodrop.pt` (does NOT overwrite the main
model) and uploaded to Drive separately.

**Run this AFTER the main training run (cell 6) is done.** ~5-6h.

In [ ]:
# 9. ABLATION: context dropout OFF (trains + auto-uploads generator_best_nodrop.pt)
import os
os.chdir(WORK_DIR)
log_file = "/kaggle/working/generator_nodrop.log"
os.system(f"python generator_train.py --context_drop 0.0 --tag _nodrop 2>&1 | tee {log_file}")
print("\nAblation training complete")

# upload the tagged checkpoint to Drive (separate from the main model)
if UPLOAD_TO_DRIVE:
    try:
        from kaggle_secrets import UserSecretsClient
        conf = UserSecretsClient().get_secret("RCLONE_CONF")
        os.makedirs("/root/.config/rclone", exist_ok=True)
        with open("/root/.config/rclone/rclone.conf", "w") as fh:
            fh.write(conf)
        if not os.path.exists("/usr/bin/rclone"):
            os.system("curl -s https://downloads.rclone.org/rclone-current-linux-amd64.zip -o /tmp/r.zip")
            os.system("cd /tmp && unzip -qo r.zip && cp rclone-*-linux-amd64/rclone /usr/bin/ && chmod +x /usr/bin/rclone")
        rc = os.system(
            "rclone copy /kaggle/working/multihop-retrieval/retrieval_v3/models/generator_best_nodrop.pt "
            f"gdrive: --drive-root-folder-id {DRIVE_FOLDER_ID} -P"
        )
        print("Uploaded generator_best_nodrop.pt to Drive" if rc == 0 else f"rclone exited {rc}")
    except Exception as e:
        print("Auto-upload skipped/failed:", e)

print("\n--- last 15 lines of ablation log (check collapse_sim stays HIGH) ---")
with open(log_file) as f:
    print("".join(f.readlines()[-15:]))

---
## Done

If auto-upload ran, `generator_best.pt` is now in your Drive `musique_data/` folder.

**Next steps:**
1. Train Model 2: run `train_model2_kaggle.ipynb`
2. Evaluate: run `eval_kaggle.ipynb`

**Decision gate:** if `collapse_sim < 0.85` AND `FULL > Graph+cosine`, G+D broke the v1/v2 collapse.